# Day 9: Loops & Iteration — When the Agent Needs to Try Again

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> Real work isn't always one shot. Sometimes the agent needs to loop.

A research agent might search → read → search again. A code agent might write → test
→ fix. Today we build agents that **iterate** until they reach a good answer.

## What You'll Build
- A research loop agent that searches multiple times
- Same pattern in all 3 frameworks
- See how loops are natural in some frameworks and forced in others

In [1]:
# ── Setup ───────────────────────────────────────────────
import sys
sys.path.insert(0, "..")
from shared import check_and_report, print_config, disable_openai_tracing, get_openai_agents_model, get_langgraph_model, get_crewai_llm
check_and_report()
disable_openai_tracing()
print("=" * 50)
print("DAY 9 SETUP COMPLETE")
print("=" * 50)
print_config()

Python: 3.12.13

All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (8 model(s) available)

DAY 9 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


## Today's Task

> **Research the top 3 facts about AI agents. Search multiple times if needed.**

The agent should:
1. Search for information
2. Evaluate if it has enough facts
3. If not, search again
4. Repeat until satisfied (max 3 iterations)

---
## Framework 1: OpenAI Agents SDK

The `max_turns` parameter on Runner limits iterations. A tool that re-invokes the
agent creates a loop.

In [2]:
from agents import Agent, Runner, function_tool
from agents.items import ItemHelpers

model = get_openai_agents_model()

# ── A search tool that returns incremental results ─────────
search_round = 0
facts_db = [
    "AI agents use LLMs to reason about tasks and decide when to use tools.",
    "The ReAct pattern (Reason + Act) is the most common agent architecture.",
    "OpenAI Agents SDK, LangGraph, and CrewAI are the top 3 Python agent frameworks.",
    "Tool calling lets agents interact with the world beyond text generation.",
    "Memory (short-term and long-term) is essential for multi-turn agent conversations.",
    "Human-in-the-loop patterns prevent agents from taking irreversible actions.",
]

@function_tool
def search_ai_agents(query: str) -> str:
    """Search for facts about AI agents. Each call returns one new fact."""
    global search_round
    if search_round < len(facts_db):
        fact = facts_db[search_round]
        search_round += 1
        return f"Fact #{search_round}: {fact}"
    return "No more facts available."

# NOTE: instructions are deliberately forceful — small models (qwen2.5:3b)
# will skip tool calls and hallucinate if you don't require them explicitly.
researcher = Agent(
    name="Research Agent",
    instructions=(
        "You are a researcher studying AI agents. "
        "You MUST call the `search_ai_agents` tool at least 3 times before answering. "
        "Do NOT state facts from memory — only use what the tool returns. "
        "After the 3rd search, compile the facts into a numbered summary."
    ),
    tools=[search_ai_agents],
    model=model,
)

print("=" * 60)
print("OPENAI AGENTS SDK — RESEARCH LOOP")
print("=" * 60)

search_round = 0  # Reset
result = await Runner.run(
    researcher,
    "Research the top 3 facts about AI agents. Search multiple times.",
    max_turns=10,  # Allow up to 10 LLM calls (tool calls count as turns)
)

# ── Print the run trace (the "process of thought") ─────────
# Runner.run() returns a RunResult whose .new_items list captures every
# step of the loop: each tool call, each tool result, each agent message.
# This is the OpenAI SDK equivalent of LangGraph's message trace.
print("\n--- RUN TRACE (each step of the loop) ---")
tool_call_count = 0
for item in result.new_items:
    if item.type == "tool_call_item":
        tool_call_count += 1
        name = getattr(item.raw_item, "name", None) or "?"
        print(f"  [turn {tool_call_count}] AGENT  -> called: {name}")
    elif item.type == "tool_call_output_item":
        out = getattr(item.raw_item, "output", "")
        print(f"             TOOL  -> returned: {str(out)[:80]}...")
    elif item.type == "message_output_item":
        text = ItemHelpers.text_message_output(item).strip()
        if text:
            print(f"  AGENT said: {text[:120]}")

print(f"\n# tool calls: {tool_call_count}  |  # total steps: {len(result.new_items)}")

print("\n--- FINAL ANSWER ---")
print(result.final_output)

OPENAI AGENTS SDK — RESEARCH LOOP

--- RUN TRACE (each step of the loop) ---
  [turn 1] AGENT  -> called: search_ai_agents
  [turn 2] AGENT  -> called: search_ai_agents
  [turn 3] AGENT  -> called: search_ai_agents
             TOOL  -> returned: ...
             TOOL  -> returned: ...
             TOOL  -> returned: ...
  AGENT said: I have compiled the facts about AI agents:

1. AI agents use LLMs to reason about tasks and decide when to use tools.
2.

# tool calls: 3  |  # total steps: 7

--- FINAL ANSWER ---
I have compiled the facts about AI agents:

1. AI agents use LLMs to reason about tasks and decide when to use tools.
2. The ReAct pattern (Reason + Act) is the most common architecture for AI agents.
3. OpenAI Agents SDK, LangGraph, and CrewAI are the top three Python agent frameworks.


---
## Framework 2: LangGraph — Natural Graph Cycles

This is where LangGraph SHINES. Loops are built into the graph — the tool node
routes back to the agent node, creating a natural cycle.

In [3]:
# NOTE: `langgraph.prebuilt.create_react_agent` is deprecated in LangGraph v1.0.
# The new import is: from langchain.agents import create_agent
# We keep the old import for compatibility with currently installed versions.
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

model = get_langgraph_model()

search_round = 0

@tool
def search_ai_agents(query: str) -> str:
    """Search for facts about AI agents. Each call returns one new fact."""
    global search_round
    facts_db = [
        "AI agents use LLMs to reason about tasks and decide when to use tools.",
        "The ReAct pattern (Reason + Act) is the most common agent architecture.",
        "OpenAI Agents SDK, LangGraph, and CrewAI are the top 3 Python agent frameworks.",
        "Tool calling lets agents interact with the world beyond text generation.",
        "Memory is essential for multi-turn agent conversations.",
        "Human-in-the-loop patterns prevent agents from taking irreversible actions.",
    ]
    if search_round < len(facts_db):
        fact = facts_db[search_round]
        search_round += 1
        return f"Fact #{search_round}: {fact}"
    return "No more facts available."

# create_react_agent AUTOMATICALLY creates a loop:
# agent -> tool -> agent -> tool -> ... -> END
agent = create_react_agent(
    model,
    tools=[search_ai_agents],
    prompt=(
        "You are a researcher studying AI agents. "
        "You MUST call `search_ai_agents` at least 3 times before answering. "
        "When you have 3 facts, compile them into a numbered summary."
    ),
)

print("=" * 60)
print("LANGGRAPH - RESEARCH LOOP (natural graph cycle)")
print("=" * 60)

search_round = 0  # Reset
result = agent.invoke({
    "messages": [HumanMessage(content="Research the top 3 facts about AI agents.")]
})

# Show the full loop trace (same format as the OpenAI SDK cell above)
tool_call_count = 0
print(f"Total messages in state: {len(result['messages'])}")
for i, msg in enumerate(result["messages"]):
    role = getattr(msg, 'type', 'unknown')
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        tool_call_count += 1
        print(f"  [turn {tool_call_count}] AGENT  -> called: {msg.tool_calls[0]['name']}")
    elif role == 'tool':
        print(f"             TOOL  -> returned: {msg.content[:80]}...")
    elif role == 'human':
        pass  # skip user input
    elif hasattr(msg, 'content') and msg.content:
        if i == len(result['messages']) - 1:
            print(f"\n--- FINAL ANSWER ---")
            print(msg.content)

print(f"\n# tool calls: {tool_call_count}  |  # messages: {len(result['messages'])}")

/var/folders/s0/2pw05fm942n5t1b5j7zzd_5m0000gn/T/ipykernel_29834/3166807291.py:32: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


LANGGRAPH - RESEARCH LOOP (natural graph cycle)
Total messages in state: 7
  [turn 1] AGENT  -> called: search_ai_agents
             TOOL  -> returned: Fact #1: AI agents use LLMs to reason about tasks and decide when to use tools....
             TOOL  -> returned: Fact #2: The ReAct pattern (Reason + Act) is the most common agent architecture....
             TOOL  -> returned: Fact #3: OpenAI Agents SDK, LangGraph, and CrewAI are the top 3 Python agent fra...
             TOOL  -> returned: Fact #4: Tool calling lets agents interact with the world beyond text generation...

--- FINAL ANSWER ---
Based on the facts retrieved:

1. Fact #1: AI agents use LLMs to reason about tasks and decide when to use tools.
2. Fact #2: The ReAct pattern (Reason + Act) is the most common agent architecture.
3. Fact #3: OpenAI Agents SDK, LangGraph, and CrewAI are the top 3 Python agent frameworks.

These three facts provide an overview of different aspects of AI agents including their reasoning capab

---
## Framework 3: CrewAI — Loop Inside a Task

CrewAI has **two ways** to create loops:

1. **Loop inside a task** (what we demo here): a single agent calls a tool
   multiple times before finishing. The loop is the agent → tool → agent → tool
   cycle within one task — same as OpenAI SDK and LangGraph.
2. **`Process.hierarchical`**: a manager agent delegates tasks and can
   *re-delegate* if the result isn't good enough. This is CrewAI's unique loop
   mechanism, but it's heavier and less reliable with small local models.

We use approach #1 with `Process.sequential` — it's the most direct comparison
to the other two frameworks. The `verbose=True` flag on each agent prints the
agent's thoughts and tool calls as they happen (that's your "process of thought").

In [4]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool as crewai_tool

llm = get_crewai_llm()

search_round = 0

@crewai_tool
def search_ai_agents(query: str) -> str:
    """Search for facts about AI agents."""
    global search_round
    # Aligned to the same 6 facts as the OpenAI SDK and LangGraph cells
    facts_db = [
        "AI agents use LLMs to reason about tasks and decide when to use tools.",
        "The ReAct pattern (Reason + Act) is the most common agent architecture.",
        "OpenAI Agents SDK, LangGraph, and CrewAI are the top 3 Python agent frameworks.",
        "Tool calling lets agents interact with the world beyond text generation.",
        "Memory is essential for multi-turn agent conversations.",
        "Human-in-the-loop patterns prevent agents from taking irreversible actions.",
    ]
    if search_round < len(facts_db):
        fact = facts_db[search_round]
        search_round += 1
        return f"Fact #{search_round}: {fact}"
    return "No more facts available."

researcher = Agent(
    role="Researcher",
    goal="Find facts about AI agents by calling search_ai_agents at least 3 times",
    backstory="You are thorough — you always search at least 3 times before concluding.",
    tools=[search_ai_agents],
    llm=llm,
    verbose=True,  # prints the agent's thoughts + tool calls
)

writer = Agent(
    role="Summary Writer",
    goal="Compile research into a clear summary",
    backstory="You turn scattered facts into coherent summaries.",
    llm=llm,
    verbose=True,
)

research_task = Task(
    description=(
        "Search for facts about AI agents. "
        "You MUST call the `search_ai_agents` tool at least 3 times. "
        "Do NOT state facts from memory — only use what the tool returns."
    ),
    expected_output="At least 3 facts about AI agents.",
    agent=researcher,
)

summary_task = Task(
    description="Compile the research findings into a clear, numbered summary.",
    expected_output="A summary with 3+ numbered facts about AI agents.",
    agent=writer,
    context=[research_task],
)

print("=" * 60)
print("CREWAI - RESEARCH LOOP (loop inside the task)")
print("=" * 60)

search_round = 0  # Reset
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, summary_task],
    process=Process.sequential,
    verbose=True,
)

# CRITICAL: kickoff_async() is a coroutine — it MUST be awaited.
# Without await, nothing runs and you just print a coroutine object.
result = await crew.kickoff_async()

print("\n--- FINAL ANSWER ---")
print(result)

CREWAI - RESEARCH LOOP (loop inside the task)


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.6                                                                                        │
│  Latest version:  1.14.7                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 693aaedb-5598-4901-9b2c-19442f9aa860                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Search for facts about AI agents. You MUST call the `search_ai_agents` tool at least 3 times. Do NOT     │
│  state facts from memory — only use what the tool returns.                                                      │
│  ID: 5d3070bf-175d-409e-89bf-4b2a5984a643                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Researcher                                                                                              │
│                                                                                                                 │
│  Task: Search for facts about AI agents. You MUST call the `search_ai_agents` tool at least 3 times. Do NOT     │
│  state facts from memory — only use what the tool returns.                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Summary Writer                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  1. Fact #1: AI agents utilize Large Language Models (LLMs) to reason about tasks and determine when to employ  │
│  specific tools.                                                                                                │
│  2. Fact #2: The ReAct pattern, consisting of a Reasoning phase followed by an Acting phase, is currently the   │
│  most prevalent agent architecture in AI research and development.                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Compile the research findings into a clear, numbered summary.                                            │
│  Agent: Summary Writer                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 693aaedb-5598-4901-9b2c-19442f9aa860                                                                       │
│  Final Output: 1. Fact #1: AI agents utilize Large Language Models (LLMs) to reason about tasks and determine   │
│  when to employ specific tools.                                                                                 │
│  2. Fact #2: The ReAct pattern, consisting of a Reasoning phase followed by an Acting phase, is currently the   │
│  most prevalent agent architecture in AI research and development.                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- FINAL ANSWER ---
1. Fact #1: AI agents utilize Large Language Models (LLMs) to reason about tasks and determine when to employ specific tools.
2. Fact #2: The ReAct pattern, consisting of a Reasoning phase followed by an Acting phase, is currently the most prevalent agent architecture in AI research and development.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Comparison: Loops

| Framework | How loops work | Natural or forced? | How to see each step |
|---|---|---|---|
| OpenAI SDK | `max_turns` + tool re-invocation | Natural | Iterate `result.new_items` |
| LangGraph | Graph cycles (tool → agent → tool) | **Most natural** | `result['messages']` trace |
| CrewAI | Loop inside a task, or `Process.hierarchical` | Semi-natural | `verbose=True` logging |

**Key insight:** LangGraph was BUILT for loops. The graph structure naturally supports
cycles — the tool node feeds back to the agent node. OpenAI SDK handles it well too:
`Runner.run()` returns a `RunResult` and every step (tool call, tool output, agent
message) is captured in `result.new_items`. CrewAI's loops are the least explicit —
you rely on `verbose=True` logging or task chaining to see what happened.

## Key Takeaway

Loops separate a demo from a useful agent. Research, retry, refine — these all require
iteration. LangGraph handles loops most naturally because graphs can have cycles.
OpenAI SDK handles them via `max_turns`. CrewAI via task chaining.

---
